# Freeside — Paired t-Test Analysis
**Pre-Test / Post-Test Quasi-Experimental Study**

This notebook:
1. Loads the Google Forms baseline survey CSV (pre-study)
2. Fetches behavioral proxy metrics from the Freeside backend (post-study)
3. Merges on `user_id`
4. Runs paired t-tests for each of the 5 metric pairs
5. Computes Cohen's d effect sizes
6. Produces publication-ready charts

---
**Statistical plan (per methodology PDF):**
- Normality: Shapiro-Wilk test (α = 0.05)
- If normal → Paired samples t-test
- If not normal → Wilcoxon signed-rank test
- Effect size: Cohen's d (small=0.2, medium=0.5, large=0.8)

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from io import StringIO

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 140,
})

## 1. Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
BACKEND_URL   = 'http://localhost:8000'
ADMIN_EMAIL   = 'ninachkheidze19@gmail.com'

# Study window (Freeside usage period)
START_DATE    = '2026-05-01'
END_DATE      = '2026-05-28'

# Path to Google Forms CSV export (pre-study baseline)
SURVEY_CSV    = 'baseline_survey.csv'
# ─────────────────────────────────────────────────────────────────────────────

# Expected columns in SURVEY_CSV (rename yours to match these):
#   user_id, pre_pps, pre_nasa_tlx, pre_psqi, pre_pss10
# Optional post-study survey (re-administer same questionnaires at end):
#   post_pps, post_nasa_tlx, post_psqi, post_pss10

## 2. Fetch cohort & behavioral metrics from Freeside backend

In [ ]:
# Get participant list
users_resp = requests.get(
    f'{BACKEND_URL}/research/users',
    params={'admin_email': ADMIN_EMAIL}
)
users_resp.raise_for_status()
user_ids = [u['id'] for u in users_resp.json()['users']]
print(f'Cohort size: N={len(user_ids)}')
print('User IDs:', user_ids)

In [ ]:
# Fetch all 5 metrics for the cohort
metrics_resp = requests.post(
    f'{BACKEND_URL}/research/export/json',
    json={
        'admin_email': ADMIN_EMAIL,
        'user_ids': user_ids,
        'start_date': START_DATE,
        'end_date': END_DATE,
    }
)
metrics_resp.raise_for_status()
behavioral_df = pd.DataFrame(metrics_resp.json()['rows'])
print(f'Behavioral data shape: {behavioral_df.shape}')
behavioral_df.head()

## 3. Load baseline survey (pre-study scores)

In [ ]:
# ── If you don't have the CSV yet, run this cell to generate dummy data ───────
# Remove this cell in the final thesis notebook and use real survey data.

np.random.seed(42)
n = len(user_ids)

dummy_survey = pd.DataFrame({
    'user_id':      user_ids,
    # PPS: 12–60, higher = more procrastination
    'pre_pps':      np.random.randint(32, 52, n),
    'post_pps':     np.random.randint(22, 42, n),
    # NASA-TLX: 0–100, higher = more cognitive load
    'pre_nasa_tlx': np.random.randint(55, 80, n),
    'post_nasa_tlx':np.random.randint(35, 60, n),
    # PSQI: 0–21, higher = worse sleep
    'pre_psqi':     np.random.randint(6,  14, n),
    'post_psqi':    np.random.randint(3,  10, n),
    # PSS-10: 0–40, higher = more stress
    'pre_pss10':    np.random.randint(18, 32, n),
    'post_pss10':   np.random.randint(10, 24, n),
})
survey_df = dummy_survey
print('Using DUMMY survey data — replace with real CSV before thesis submission')
survey_df.head()

In [ ]:
# ── When your real CSV is ready, comment out the dummy block above
# and uncomment this:

# survey_df = pd.read_csv(SURVEY_CSV)
# print(f'Survey data shape: {survey_df.shape}')
# survey_df.head()

## 4. Merge datasets

In [ ]:
merged = survey_df.merge(behavioral_df, on='user_id', how='inner')
print(f'Merged shape: {merged.shape}  (participants with both survey + behavioral data)')

if len(merged) < len(survey_df):
    missing = set(survey_df['user_id']) - set(behavioral_df['user_id'])
    print(f'WARNING: {len(missing)} participant(s) in survey but not in Freeside data: {missing}')

merged.head()

## 5. Statistical analysis functions

In [ ]:
def cohens_d(pre: np.ndarray, post: np.ndarray) -> float:
    """Cohen's d for paired samples (pre − post / SD_pre)."""
    diff = pre - post
    return float(np.mean(diff) / np.std(pre, ddof=1))


def effect_label(d: float) -> str:
    a = abs(d)
    if a < 0.2:  return 'negligible'
    if a < 0.5:  return 'small'
    if a < 0.8:  return 'medium'
    return 'large'


def run_paired_test(label: str, pre: pd.Series, post: pd.Series,
                    hypothesis: str = 'pre > post') -> dict:
    """
    Runs Shapiro-Wilk normality check, then either a paired t-test or
    Wilcoxon signed-rank test depending on normality. Returns a result dict.
    """
    pre_arr  = pre.dropna().values.astype(float)
    post_arr = post.dropna().values.astype(float)

    # Align by index in case of missing values
    combined = pd.DataFrame({'pre': pre, 'post': post}).dropna()
    pre_arr  = combined['pre'].values.astype(float)
    post_arr = combined['post'].values.astype(float)
    n        = len(pre_arr)

    # Shapiro-Wilk (requires n >= 3)
    if n >= 3:
        _, p_shapiro = stats.shapiro(pre_arr - post_arr)
        normal = p_shapiro > 0.05
    else:
        p_shapiro, normal = None, True

    # Choose test
    if normal:
        t_stat, p_value = stats.ttest_rel(pre_arr, post_arr)
        test_name = 'Paired t-test'
        stat_name = 't'
        stat_val  = t_stat
    else:
        w_stat, p_value = stats.wilcoxon(pre_arr, post_arr)
        test_name = 'Wilcoxon'
        stat_name = 'W'
        stat_val  = w_stat

    d = cohens_d(pre_arr, post_arr)

    result = {
        'metric':       label,
        'n':            n,
        'pre_mean':     round(float(np.mean(pre_arr)),  3),
        'post_mean':    round(float(np.mean(post_arr)), 3),
        'mean_change':  round(float(np.mean(pre_arr - post_arr)), 3),
        'test':         test_name,
        stat_name:      round(float(stat_val), 3),
        'p_value':      round(float(p_value), 4),
        'significant':  p_value < 0.05,
        'cohens_d':     round(d, 3),
        'effect_size':  effect_label(d),
        'p_shapiro':    round(float(p_shapiro), 4) if p_shapiro is not None else 'n/a',
        'hypothesis':   hypothesis,
    }
    return result


print('Functions ready.')

## 6. Run t-tests — all 5 metric pairs

| RQ | Survey instrument | Behavioral proxy | Hypothesis |
|---|---|---|---|
| RQ1 | PPS (procrastination) | TCR % | Pre PPS > Post PPS |
| RQ2 | NASA-TLX (cognitive load) | Reroute % | Pre TLX > Post TLX |
| RQ3 | PSQI (sleep) | avg_rested_score | Pre PSQI > Post PSQI |
| RQ4 | PPS (initiation delay) | avg_initiation_delay | Pre delay > Post delay |
| RQ5 | PSS-10 (stress/burnout) | overall_avg_energy | Pre PSS > Post PSS |

In [ ]:
results = []

# RQ1 — Procrastination (PPS vs TCR)
results.append(run_paired_test(
    'RQ1: PPS (pre vs post)',
    merged['pre_pps'], merged['post_pps'],
    hypothesis='post_pps < pre_pps (less procrastination after Freeside)'
))

# RQ2 — Cognitive Load (NASA-TLX vs reroute_percentage)
results.append(run_paired_test(
    'RQ2: NASA-TLX (pre vs post)',
    merged['pre_nasa_tlx'], merged['post_nasa_tlx'],
    hypothesis='post_nasa_tlx < pre_nasa_tlx (lower perceived load)'
))

# RQ3 — Sleep (PSQI vs avg_rested_score)
results.append(run_paired_test(
    'RQ3: PSQI (pre vs post)',
    merged['pre_psqi'], merged['post_psqi'],
    hypothesis='post_psqi < pre_psqi (better sleep quality)'
))

# RQ4 — PSS-10 (stress/burnout)
results.append(run_paired_test(
    'RQ4: PSS-10 (pre vs post)',
    merged['pre_pss10'], merged['post_pss10'],
    hypothesis='post_pss10 < pre_pss10 (lower perceived stress)'
))

# RQ5 — Behavioral proxy: initiation delay (pre/post periods)
# Split the study window in half and compare first vs second half delay
# If you have two separate export calls, use those instead.
# For now, compare PPS as the paired instrument for procrastination delay.

results_df = pd.DataFrame(results)
results_df

## 7. Results summary table

In [ ]:
summary = results_df[['metric', 'n', 'pre_mean', 'post_mean', 'mean_change',
                        'test', 'p_value', 'significant', 'cohens_d', 'effect_size']].copy()

summary['significant'] = summary['significant'].map({True: '✓ YES', False: '✗ no'})

print('=' * 90)
print('FREESIDE THESIS — PAIRED COMPARISON RESULTS')
print('=' * 90)
print(summary.to_string(index=False))
print()
print('Significance threshold: α = 0.05 | Effect: small≥0.2, medium≥0.5, large≥0.8')

## 8. Publication chart — Pre/Post comparison bar chart

In [ ]:
metrics_to_plot = [
    ('pre_pps',      'post_pps',      'PPS\n(Procrastination)', '#4648d4'),
    ('pre_nasa_tlx', 'post_nasa_tlx', 'NASA-TLX\n(Cognitive Load)', '#7c3aed'),
    ('pre_psqi',     'post_psqi',     'PSQI\n(Sleep Quality)', '#0891b2'),
    ('pre_pss10',    'post_pss10',    'PSS-10\n(Stress)', '#dc2626'),
]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(14, 5))
fig.suptitle('Freeside — Pre vs Post Intervention\nPaired Comparison', fontsize=13, fontweight='bold', y=1.01)

for ax, (pre_col, post_col, label, color) in zip(axes, metrics_to_plot):
    pre_vals  = merged[pre_col].dropna()
    post_vals = merged[post_col].dropna()

    bars = ax.bar(['Pre', 'Post'], [pre_vals.mean(), post_vals.mean()],
                  color=[color + 'aa', color], edgecolor=color, linewidth=1.5,
                  error_kw=dict(ecolor='#333', capsize=5))

    # Error bars (±1 SEM)
    ax.errorbar(['Pre', 'Post'],
                [pre_vals.mean(), post_vals.mean()],
                yerr=[pre_vals.sem(), post_vals.sem()],
                fmt='none', color='#333', capsize=5, linewidth=1.5)

    # Individual participant lines
    combined = pd.DataFrame({pre_col: merged[pre_col], post_col: merged[post_col]}).dropna()
    for _, row in combined.iterrows():
        ax.plot([0, 1], [row[pre_col], row[post_col]],
                color=color, alpha=0.25, linewidth=1, zorder=2)

    # p-value annotation
    row_res = results_df[results_df['metric'].str.contains(label.split('\\n')[0].strip())]
    if not row_res.empty:
        p = row_res.iloc[0]['p_value']
        p_str = f'p={p:.3f}' + (' *' if p < 0.05 else '')
        ax.set_title(p_str, fontsize=9, color='#444')

    ax.set_xlabel(label, fontsize=10, fontweight='semibold')
    ax.set_ylabel('Mean Score')

plt.tight_layout()
plt.savefig('freeside_pre_post_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: freeside_pre_post_comparison.png')

## 9. Behavioral proxy chart — Rolling energy trends (SWBBS)

In [ ]:
# Fetch SWBBS detail for each participant to plot energy trajectories
COLORS = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Rolling 7-Day Energy Score — All Participants\n(SWBBS proxy for PSS-10/OLBI burnout)',
             fontsize=12, fontweight='bold')

for i, uid in enumerate(user_ids[:10]):  # cap at 10 for legibility
    # Re-fetch individual SWBBS detail
    resp = requests.post(f'{BACKEND_URL}/research/export/json', json={
        'admin_email': ADMIN_EMAIL,
        'user_ids': [uid],
        'start_date': START_DATE,
        'end_date': END_DATE,
    })
    row = resp.json()['rows'][0] if resp.ok and resp.json()['rows'] else None
    if not row:
        continue

    trend = behavioral_df[behavioral_df['user_id'] == uid]['overall_avg_energy'].values
    # Note: for per-day trajectory, use the daily_trend_array from get_wellbeing_burnout_score directly
    # Here we plot overall_avg_energy as a single point per participant for overview
    ax.scatter(i, trend[0] if len(trend) else 0,
               color=COLORS[i % 10], s=80, zorder=3,
               label=f'P{i+1:02d}')
    if row['burnout_flag']:
        ax.annotate('⚠', (i, trend[0] if len(trend) else 0),
                    textcoords='offset points', xytext=(6, 4),
                    color='red', fontsize=11)

ax.axhline(3.0, color='#dc2626', linestyle='--', linewidth=1.2, label='Burnout threshold (energy < 3)')
ax.set_xticks(range(len(user_ids[:10])))
ax.set_xticklabels([f'P{i+1:02d}' for i in range(len(user_ids[:10]))])
ax.set_ylabel('Avg Energy Score (1–5)')
ax.set_ylim(0, 5.5)
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('freeside_energy_trends.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: freeside_energy_trends.png')

## 10. Export final results table to CSV

In [ ]:
# Full merged dataset — ready for supervisor review
merged.to_csv('freeside_full_dataset.csv', index=False)
print('Saved: freeside_full_dataset.csv')

# Statistical results table
results_df.to_csv('freeside_statistical_results.csv', index=False)
print('Saved: freeside_statistical_results.csv')

print()
print('Files for thesis appendix:')
print('  freeside_full_dataset.csv          → Appendix C: Raw data')
print('  freeside_statistical_results.csv   → Appendix C: Statistical output')
print('  freeside_pre_post_comparison.png   → Chapter 5: Figure 1')
print('  freeside_energy_trends.png         → Chapter 5: Figure 2')